# B2-022 — Session 1: Multivariate Gaussians

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

This session upgrades the one-variable Gaussian from Book 1 F5 to vectors.
You will learn to read a mean vector and a covariance matrix, compute a determinant for the symmetric positive-definite matrices that appear as covariances, evaluate a log-density stably with `torch.linalg.slogdet` and `torch.linalg.solve`, draw samples with a Cholesky or spectral factor, and fit a Gaussian by maximum likelihood.
Every later session (KL divergence, autoencoder latents, the VAE) is built from the objects defined here.

## 1. Mean vector and covariance matrix

Let $x=(x_1,\dots,x_d)$ be a random vector.
Its **mean vector** is $\mu=E[x]$, computed coordinate by coordinate: $\mu_i=E[x_i]$.
Its **covariance matrix** $\Sigma$ is the $d\times d$ table of the Book 1 scalar covariances,
$$\Sigma_{ij}=\operatorname{Cov}(x_i,x_j)=E[(x_i-\mu_i)(x_j-\mu_j)].$$
In matrix form this is $\Sigma=E[(x-\mu)(x-\mu)^\top]$: the expectation of an outer product, taken entry by entry.

Three facts follow directly from the scalar definition.
First, $\Sigma$ is **symmetric** because $\operatorname{Cov}(x_i,x_j)=\operatorname{Cov}(x_j,x_i)$.
Second, the **diagonal** holds the variances, $\Sigma_{ii}=\operatorname{Var}(x_i)\ge 0$.
Third, for any fixed vector $a$, the scalar $a^\top x$ has variance $a^\top\Sigma a$, so $a^\top\Sigma a\ge 0$ for every $a$.
A symmetric matrix with $a^\top\Sigma a>0$ for every nonzero $a$ is called **symmetric positive definite (SPD)**.
By the Book 1 spectral theorem, a symmetric matrix is SPD exactly when all of its eigenvalues are positive.
Every covariance in this unit's densities is SPD; a covariance with a zero eigenvalue describes data squashed onto a lower-dimensional flat, and it has no density in $\mathbb R^d$.

From data rows $x^{(1)},\dots,x^{(n)}$ stacked as an `(n,d)` tensor `X`, the sample mean is `X.mean(dim=0)` with shape `(d,)`.
The sample covariance with the **$1/n$ convention** is `(X - m).T @ (X - m) / n` with shape `(d,d)`.
Book 1 C9 used the $1/(n-1)$ convention for PCA; both are legitimate, and the difference is a scalar factor $n/(n-1)$.
In this unit the maximum-likelihood estimate (Section 7) uses $1/n$, so every coded covariance in this unit says which denominator it uses.

Shape ledger for one batch: `X (n,d)` → `m (d,)` → `X - m (n,d)` by broadcasting → `(X - m).T @ (X - m) (d,d)`.

**Checkpoint 1A.** For three 2-D points $(0,0),(2,0),(1,3)$, compute the mean vector and the $1/n$ covariance matrix.

**Checkpoint 1B.** Why must every covariance matrix satisfy $a^\top\Sigma a\ge0$?

**Collected answers.** The mean is $(1,1)$; centered rows are $(-1,-1),(1,-1),(0,2)$, so $\Sigma=\frac13\begin{pmatrix}2&0\\0&6\end{pmatrix}=\begin{pmatrix}2/3&0\\0&2\end{pmatrix}$. Because $a^\top\Sigma a=\operatorname{Var}(a^\top x)$ and a variance is never negative.

## 2. Determinants of SPD matrices and `slogdet`

Book 1 F6 deliberately stopped short of determinants, so this unit defines exactly the part it needs.
For a **symmetric** matrix $\Sigma$ with spectral decomposition $\Sigma=Q\Lambda Q^\top$ (orthonormal $Q$, eigenvalues $\lambda_1,\dots,\lambda_d$ on the diagonal of $\Lambda$), define
$$\det\Sigma=\lambda_1\lambda_2\cdots\lambda_d .$$
Geometrically, $\Sigma$ stretches the unit cube along its eigenvectors by $\lambda_1,\dots,\lambda_d$, so the determinant is the factor by which it scales volume.
For an SPD matrix every $\lambda_i>0$, so $\det\Sigma>0$ and $\log\det\Sigma=\sum_i\log\lambda_i$ is well defined.
Two special cases are worth memorizing.
A diagonal matrix has its diagonal entries as eigenvalues, so its determinant is the product of the diagonal.
A symmetric $2\times2$ matrix $\begin{pmatrix}a&b\\b&c\end{pmatrix}$ has characteristic polynomial $\lambda^2-(a+c)\lambda+(ac-b^2)$; its two roots multiply to the constant term, so
$$\det\begin{pmatrix}a&b\\b&c\end{pmatrix}=ac-b^2 .$$
(For general square matrices the same name has a broader definition, which this unit never needs.)

**Why `slogdet`.** A product of many eigenvalues overflows or underflows quickly: in $d=200$ with every eigenvalue equal to $0.01$, the determinant is $10^{-400}$, which is zero in float64.
Its logarithm, $-921.03$, is perfectly ordinary.
`torch.linalg.slogdet(S)` returns a pair `(sign, logabsdet)` with $\det S=\text{sign}\cdot e^{\text{logabsdet}}$, computed without forming the product.
For an SPD covariance the sign must be `+1`; a sign of `-1` or `0` is a diagnostic that the matrix is not a valid covariance.
A positive sign alone does not prove SPD (two negative eigenvalues also give a positive product), so code that must reject invalid input also checks symmetry or relies on `torch.linalg.cholesky`, which fails on non-SPD input (Section 5).
The banned shortcut in this unit is `torch.det` followed by `torch.log`: it reproduces the overflow you just saw.

**Checkpoint 2A.** Compute $\det\begin{pmatrix}4&2\\2&3\end{pmatrix}$ two ways: with $ac-b^2$, and as the product of the eigenvalues $(7\pm\sqrt{17})/2$.

**Checkpoint 2B.** A symmetric matrix has eigenvalues $3,-1,-2$. What does `slogdet` return for its sign, and is it a valid covariance?

**Collected answers.** $4\cdot3-2^2=8$, and $\frac{(7+\sqrt{17})(7-\sqrt{17})}{4}=\frac{49-17}{4}=8$. The product is $6>0$, so the sign is `+1`, yet the matrix is not SPD (it has negative eigenvalues) and is not a valid covariance; a positive sign is necessary, not sufficient.

In [ ]:
import math
import torch

sigma = torch.tensor([[4.0, 2.0], [2.0, 3.0]], dtype=torch.float64)
eigenvalues = torch.linalg.eigvalsh(sigma)          # ascending, real, for symmetric input
sign, logabsdet = torch.linalg.slogdet(sigma)
print("eigenvalues:", eigenvalues.tolist())
print("product of eigenvalues:", eigenvalues.prod().item(), " 2x2 formula:", 4.0 * 3.0 - 2.0 ** 2)
print("slogdet:", sign.item(), logabsdet.item(), " log 8 =", math.log(8.0))
assert torch.allclose(logabsdet, torch.tensor(math.log(8.0), dtype=torch.float64), atol=1e-12, rtol=1e-12)

tiny = 0.01 * torch.eye(200, dtype=torch.float64)
print("naive det:", torch.linalg.eigvalsh(tiny).prod().item(), " slogdet:", torch.linalg.slogdet(tiny))

## 3. Density and log-density

The $d$-dimensional Gaussian $N(\mu,\Sigma)$ with SPD covariance has density
$$p(x)=(2\pi)^{-d/2}(\det\Sigma)^{-1/2}\exp\!\Big(-\tfrac12 (x-\mu)^\top\Sigma^{-1}(x-\mu)\Big).$$
The scalar $q(x)=(x-\mu)^\top\Sigma^{-1}(x-\mu)$ is the **quadratic form** (also called the squared Mahalanobis distance).
The log-density is the sum of a **normalizer** and the quadratic-form term:
$$\log p(x)=-\tfrac12\big(d\log 2\pi+\log\det\Sigma\big)-\tfrac12 q(x).$$

**Why this formula is the right extension.** Start with the diagonal case $\Sigma=\operatorname{diag}(\sigma_1^2,\dots,\sigma_d^2)$.
Then $\det\Sigma=\prod_i\sigma_i^2$ and $q(x)=\sum_i (x_i-\mu_i)^2/\sigma_i^2$, so the formula is exactly the product of the $d$ Book 1 univariate densities $N(\mu_i,\sigma_i^2)$: independent coordinates multiply.
For a full $\Sigma=Q\Lambda Q^\top$, change to eigen-coordinates $y=Q^\top(x-\mu)$.
Because $\Sigma^{-1}=Q\Lambda^{-1}Q^\top$, the quadratic form becomes $q=\sum_i y_i^2/\lambda_i$, and $\det\Sigma=\prod_i\lambda_i$.
So a full-covariance Gaussian is a product of independent univariate Gaussians with variances $\lambda_i$ along the eigenvector axes; the rotation $Q$ preserves lengths and volumes (we state this property of orthonormal matrices), so the normalizer does not change.

**Computing $q$ without an inverse.** Never form $\Sigma^{-1}$ explicitly.
Solve the linear system $\Sigma v=x-\mu$ with `torch.linalg.solve(sigma, diff)` (the Torch twin of the `np.linalg.solve` used in Book 1 C2) and take the dot product: $q=(x-\mu)^\top v$.
For a batch `X (n,d)`, `diff = X - mu` is `(n,d)`, `torch.linalg.solve(sigma, diff.T)` is `(d,n)`, and `(diff * solved.T).sum(dim=1)` is the `(n,)` vector of quadratic forms.
Solving is cheaper and more accurate than inverting, and it is the pattern p06 requires.

**Checkpoint 3A.** Write $\log p(x)$ for $N(0,I_d)$ and evaluate it at $x=0$ for $d=2$.

**Checkpoint 3B.** In a batched log-density, `diff` has shape `(5,3)`. What are the shapes of `torch.linalg.solve(sigma, diff.T)` and of the final quadratic-form vector?

**Collected answers.** $\log p(x)=-\frac d2\log2\pi-\frac12\lVert x\rVert^2$; at $x=0$, $d=2$ it is $-\log 2\pi\approx-1.837877$. The solve returns `(3,5)` and the quadratic forms are `(5,)`.

## 4. Covariance eigen-geometry: ellipse axes

The level sets $\{x: q(x)=c\}$ of a Gaussian density are ellipses (ellipsoids in $d>2$).
In eigen-coordinates the level set reads $\sum_i y_i^2/\lambda_i=c$, which is an axis-aligned ellipse with **semi-axis lengths $\sqrt{c\lambda_i}$**.
Rotating back with $Q$, the ellipse is centered at $\mu$, its axes point along the **eigenvectors** of $\Sigma$, and the semi-axis along eigenvector $v_i$ has length $\sqrt{c\,\lambda_i}$.
For the "one-standard-deviation" contour $c=1$ the semi-axes are $\sqrt{\lambda_i}$, the standard deviations of the data along each principal direction.
This is the same eigen-decomposition that Book 1 C9 used for PCA: the first principal direction is the longest ellipse axis.

**Worked reading.** For $\Sigma=\begin{pmatrix}3&1\\1&3\end{pmatrix}$, the eigenvalues are $4$ and $2$ with eigenvectors $(1,1)/\sqrt2$ and $(1,-1)/\sqrt2$.
The $c=1$ ellipse has semi-major axis $2$ along the $45^\circ$ diagonal and semi-minor axis $\sqrt2$ along the anti-diagonal.
Positive off-diagonal covariance tilts the ellipse toward the $y=x$ direction; negative covariance would tilt it toward $y=-x$; zero covariance with unequal variances gives an axis-aligned ellipse.

Two common errors live here.
Reading the eigenvalue itself as the axis length forgets the square root: variance is squared length.
Reading a **row** of the matrix returned by `torch.linalg.eigh` as an eigenvector is a transpose error: the eigenvectors are the **columns**, and `eigh` returns eigenvalues in ascending order, so the principal axis is the last column.

**Checkpoint 4A.** For $\Sigma=\operatorname{diag}(9,1)$, give the semi-axis lengths and directions of the $c=1$ contour.

**Checkpoint 4B.** `evals, Q = torch.linalg.eigh(S)` returns `evals=[2.,4.]`. Which slice of `Q` is the principal (longest-axis) direction?

**Collected answers.** Semi-axes $3$ along $(1,0)$ and $1$ along $(0,1)$. The last column, `Q[:, 1]` (or `Q[:, -1]`), because `eigh` sorts eigenvalues in ascending order and stores eigenvectors as columns.

## 5. The rule $\operatorname{Cov}(Az+b)=A\operatorname{Cov}(z)A^\top$ and Cholesky sampling

**Derivation from the scalar definition.** Let $z$ have mean $m$ and covariance $C$, and let $x=Az+b$ for a fixed matrix $A$ and vector $b$.
Linearity of expectation gives $E[x]=Am+b$, so $x-E[x]=A(z-m)$.
Coordinate $i$ of that difference is $\sum_k A_{ik}(z_k-m_k)$, and expanding the product of two such sums,
$$\operatorname{Cov}(x_i,x_j)=E\Big[\sum_k A_{ik}(z_k-m_k)\sum_l A_{jl}(z_l-m_l)\Big]=\sum_{k,l}A_{ik}C_{kl}A_{jl}=(ACA^\top)_{ij}.$$
So $\operatorname{Cov}(Az+b)=A\,\operatorname{Cov}(z)\,A^\top$; the shift $b$ moves the mean and never changes the covariance.
We also state, without proof, that an affine map of a Gaussian vector is again Gaussian.

**Sampling recipe.** If $z\sim N(0,I)$ and $LL^\top=\Sigma$, then $x=\mu+Lz$ has mean $\mu$ and covariance $LIL^\top=\Sigma$, so $x\sim N(\mu,\Sigma)$.
Any factor $L$ with $LL^\top=\Sigma$ works. Two standard choices:

* **Cholesky.** `torch.linalg.cholesky(sigma)` returns the unique lower-triangular `L` with positive diagonal and `L @ L.T == sigma` (up to rounding). We state this property and use it as an API; it exists exactly for SPD input, and the call raises an error otherwise, which makes it a practical SPD test.
* **Spectral square root.** With `evals, Q = torch.linalg.eigh(sigma)`, the factor `S = Q @ torch.diag(evals.sqrt())` satisfies $SS^\top=Q\Lambda^{1/2}\Lambda^{1/2}Q^\top=\Sigma$.

The two factors are different matrices, so the same standard-normal draws map to different samples, but both produce the same distribution.

**Row convention in code.** Data tensors store one sample per **row**. Draw `z = torch.randn(n, d, generator=g, dtype=torch.float64)` and compute `x = mu + z @ L.T`; row $i$ is then $\mu+Lz^{(i)}$.
Writing `z @ L` instead silently samples from $N(\mu, L^\top L)$, which is a different covariance whenever $L$ is not symmetric.

**Checkpoint 5A.** If $z\sim N(0,I_2)$ and $x=\begin{pmatrix}2&0\\1&1\end{pmatrix}z+\begin{pmatrix}5\\-5\end{pmatrix}$, what are $E[x]$ and $\operatorname{Cov}(x)$?

**Checkpoint 5B.** Why does `z @ L` (instead of `z @ L.T`) give the wrong covariance for a Cholesky factor?

**Collected answers.** $E[x]=(5,-5)$ and $\operatorname{Cov}(x)=AA^\top=\begin{pmatrix}4&2\\2&2\end{pmatrix}$. Row-wise `z @ L` computes $L^\top z$ for each sample, whose covariance is $L^\top L$, which differs from $LL^\top=\Sigma$ unless $L$ is symmetric.

## 6. Worked laboratory: 2-D density and Cholesky sampling

Take $\mu=(1,-1)$, $\Sigma=\begin{pmatrix}4&2\\2&3\end{pmatrix}$, and the point $x=(3,0)$.

**Step 1: determinant.** $\det\Sigma=4\cdot3-2^2=8$, so $\log\det\Sigma=\log8=2.079442$.

**Step 2: quadratic form by solving.** $x-\mu=(2,1)$. Solve $\Sigma v=(2,1)$: from $4v_1+2v_2=2$ and $2v_1+3v_2=1$ we get $v=(1/2,0)$. Then $q=(2,1)\cdot(1/2,0)=1$.

**Step 3: log-density.** $\log p(x)=-\tfrac12(2\log2\pi+\log8)-\tfrac12\cdot1=-1.837877-1.039721-0.5=-3.377598$.

**Step 4: Cholesky factor by hand.** Write $L=\begin{pmatrix}\ell_{11}&0\\\ell_{21}&\ell_{22}\end{pmatrix}$. Matching $LL^\top=\Sigma$ entry by entry: $\ell_{11}^2=4\Rightarrow\ell_{11}=2$; $\ell_{21}\ell_{11}=2\Rightarrow\ell_{21}=1$; $\ell_{21}^2+\ell_{22}^2=3\Rightarrow\ell_{22}=\sqrt2$.

**Step 5: one sample.** For the standard-normal draw $z=(0.5,-1)$: $Lz=(1,\,0.5-\sqrt2)$, so $x=\mu+Lz=(2,\,-0.5-\sqrt2)=(2,-1.914214)$.

**Step 6: certify the sampler statistically.** With 100,000 seeded draws the $1/n$ sample covariance should land near $\Sigma$, but not exactly: Monte Carlo error shrinks like $1/\sqrt n$, so the check needs an explicit tolerance. With seed `20260927` the first cell below prints a mean near $(1.0011,-0.9994)$ and covariance entries within about $0.02$ of $\Sigma$. A tolerance of $0.05$ is generous for $n=10^5$; a tolerance of $10^{-6}$ would be a category error.

The cells below reproduce Steps 1–5 exactly and Step 6 statistically, and they check the spectral factor as a second valid sampler.

**Checkpoint 6A.** Repeat Step 2 for $x=(1,2)$ with the same $\mu,\Sigma$ and give $q$.

**Checkpoint 6B.** Why is an exact equality test on a Monte Carlo sample covariance the wrong certificate?

**Collected answers.** $x-\mu=(0,3)$; solving $4v_1+2v_2=0$, $2v_1+3v_2=3$ gives $v=(-3/4,3/2)$, so $q=0\cdot(-3/4)+3\cdot3/2=9/2$. Sample statistics fluctuate with standard error of order $1/\sqrt n$, so a correct sampler fails an exact test; the right certificate states a tolerance tied to $n$.

In [ ]:
import math
import torch

mu = torch.tensor([1.0, -1.0], dtype=torch.float64)
sigma = torch.tensor([[4.0, 2.0], [2.0, 3.0]], dtype=torch.float64)
x = torch.tensor([3.0, 0.0], dtype=torch.float64)

sign, logabsdet = torch.linalg.slogdet(sigma)
diff = x - mu
quad = diff @ torch.linalg.solve(sigma, diff)
log_density = -0.5 * (2 * math.log(2 * math.pi) + logabsdet) - 0.5 * quad
print("sign, log det, q, log p:", sign.item(), round(logabsdet.item(), 6), quad.item(), round(log_density.item(), 6))
assert math.isclose(log_density.item(), -3.377598, abs_tol=1e-6, rel_tol=0.0)

L = torch.linalg.cholesky(sigma)
print("L =", L.tolist())
assert torch.allclose(L @ L.T, sigma, atol=1e-12, rtol=1e-12)
one_sample = mu + L @ torch.tensor([0.5, -1.0], dtype=torch.float64)
print("mu + L z =", one_sample.tolist())

In [ ]:
import torch

torch.manual_seed(20260927)
mu = torch.tensor([1.0, -1.0], dtype=torch.float64)
sigma = torch.tensor([[4.0, 2.0], [2.0, 3.0]], dtype=torch.float64)
generator = torch.Generator().manual_seed(20260927)
z = torch.randn(100_000, 2, generator=generator, dtype=torch.float64)

for name, factor in [("cholesky", torch.linalg.cholesky(sigma)),
                     ("spectral", torch.linalg.eigh(sigma)[1] @ torch.diag(torch.linalg.eigh(sigma)[0].sqrt()))]:
    samples = mu + z @ factor.T                      # one sample per row
    mean = samples.mean(dim=0)
    cov = (samples - mean).T @ (samples - mean) / samples.shape[0]   # 1/n convention
    print(name, "mean", [round(v, 4) for v in mean.tolist()], "cov", [[round(v, 3) for v in row] for row in cov.tolist()])
    assert torch.allclose(mean, mu, atol=0.05, rtol=0.0)
    assert torch.allclose(cov, sigma, atol=0.05, rtol=0.0)

## 7. Diagonal versus full covariance, and maximum likelihood

**Parameter counts.** A full covariance in $d$ dimensions has $d(d+1)/2$ free numbers (the upper triangle); a diagonal covariance has $d$.
For $d=8$ that is $36$ versus $8$.
A diagonal Gaussian cannot represent correlation: its ellipses are axis-aligned.
For a Gaussian, zero covariance between coordinates is the same as independence, because Section 3 showed the density then factorizes into univariate densities.
Choose full covariance when correlations matter and you have enough data to estimate them; choose diagonal when $d$ is large relative to $n$, when you need cheap $O(d)$ log-densities, or (as in the VAE) when the model's other parts are responsible for correlations.

**Maximum likelihood (MLE).** Given independent rows $x^{(1)},\dots,x^{(n)}$, the log-likelihood is $\ell(\mu,\Sigma)=\sum_k\log p(x^{(k)})$, and the MLE is the parameter value that maximizes it.
For a **diagonal** covariance, the log-density is a sum over coordinates, so $\ell$ splits into $d$ separate univariate problems, one per coordinate $j$:
$$\ell_j(m,v)=-\tfrac n2\log(2\pi v)-\frac{1}{2v}\sum_k (x^{(k)}_j-m)^2,\qquad v=\sigma_j^2>0 .$$
Here is the one-coordinate derivation with Calculus AB tools.
Fix $v$; $\ell_j$ is a downward parabola in $m$ with $\partial\ell_j/\partial m=\frac1v\sum_k(x^{(k)}_j-m)$, which is zero exactly at $m=\bar x_j$, the sample mean; this holds for every $v$, so $\hat m_j=\bar x_j$.
Now set $S=\sum_k(x^{(k)}_j-\bar x_j)^2$ and study $g(v)=-\frac n2\log v-\frac{S}{2v}$ on $v>0$: $g'(v)=-\frac{n}{2v}+\frac{S}{2v^2}=\frac{S-nv}{2v^2}$, positive for $v<S/n$ and negative for $v>S/n$.
So $g$ rises then falls, and the unique maximizer is $\hat v_j=S/n$, the $1/n$ variance.
**Where positivity is needed.** If $S=0$ (fewer than two rows, or all values of coordinate $j$ equal), then $g(v)=-\frac n2\log v$ increases without bound as $v\to0^+$: the likelihood is unbounded and no MLE exists. So the diagonal MLE needs, for every coordinate, $n\ge2$ and not all values equal.

**Full covariance (stated, not derived).** The full-covariance MLE is $\hat\mu=\bar x$ and $\hat\Sigma=\frac1n\sum_k(x^{(k)}-\bar x)(x^{(k)}-\bar x)^\top$. Its proof needs matrix calculus beyond this course. It is a valid MLE only when $\hat\Sigma$ is positive definite, which happens exactly when the centered rows span $\mathbb R^d$; that requires $n>d$. Rank-deficient data (for example $n\le d$, or rows lying on a line in $\mathbb R^2$) has no unrestricted MLE: shrinking the variance across the flat drives the likelihood to infinity.

**Checkpoint 7A.** Rows $(1,4),(3,4),(2,4)$: give the diagonal MLE, and say whether it exists.

**Checkpoint 7B.** A team fits a full-covariance Gaussian to $n=5$ rows in $d=8$. What goes wrong, and name one remedy from this section.

**Collected answers.** Coordinate 1 has mean $2$ and $1/n$ variance $2/3$; coordinate 2 has all values equal ($S=0$), so its variance MLE would be $0$ and the likelihood is unbounded: no diagonal MLE exists. With $n\le d$ the centered rows cannot span $\mathbb R^8$, so $\hat\Sigma$ is singular and no MLE exists; use a diagonal covariance (if each coordinate has $n\ge2$ unequal values) or collect more than $8$ rows.

In [ ]:
import torch

rows = torch.tensor([[0.0, 1.0, 2.0], [2.0, 1.5, 2.0], [1.0, -0.5, 4.0], [3.0, 2.0, 0.0]], dtype=torch.float64)
n, d = rows.shape
mean = rows.mean(dim=0)
centered = rows - mean
diag_mle = (centered ** 2).sum(dim=0) / n                 # one 1/n variance per coordinate
full_mle = centered.T @ centered / n                      # stated full-covariance MLE
print("mean:", mean.tolist())
print("diagonal MLE variances:", diag_mle.tolist())
print("full MLE eigenvalues:", torch.linalg.eigvalsh(full_mle).tolist())
assert torch.allclose(torch.diagonal(full_mle), diag_mle, atol=1e-12, rtol=1e-12)
# n = 4 > d = 3 here, and every eigenvalue is positive, so the full MLE exists.

## 8. Common pitfalls and Exam connections

**Common pitfalls (each with its fix).**

1. *Inverting instead of solving.* `diff @ torch.inverse(sigma) @ diff` is slower and less accurate. Fix: `diff @ torch.linalg.solve(sigma, diff)`.
2. *Overflowing determinant.* `torch.log(torch.det(sigma))` returns `-inf` for the $200$-dimensional example in Section 2. Fix: `torch.linalg.slogdet(sigma)` and check the sign.
3. *Wrong normalizer power.* Writing $(2\pi)^{-d}$ or $(\det\Sigma)^{-1}$ instead of $(2\pi)^{-d/2}(\det\Sigma)^{-1/2}$. Fix: derive from the diagonal case, where each coordinate contributes $(2\pi\sigma_i^2)^{-1/2}$.
4. *Row/column factor confusion.* `mu + z @ L` samples covariance $L^\top L$. Fix: `mu + z @ L.T` with rows as samples.
5. *Eigenvalue as axis length.* The semi-axis is $\sqrt{\lambda}$, not $\lambda$; and `eigh` eigenvectors are columns.
6. *Denominator drift.* Mixing $1/(n-1)$ from C9 with the $1/n$ MLE. Fix: name the convention in the code comment and the report.
7. *Forgetting existence conditions.* Reporting a covariance MLE for rank-deficient data. Fix: check $n>d$ and positive eigenvalues (or a successful Cholesky).

**Exam connections.** Round 2 style tasks ask you to (a) evaluate a log-density term exactly with integer data, as a reduced fraction; (b) read axis lengths and directions from a covariance; (c) implement a batched log-density under an API ban (no inverse, no `torch.distributions`) and match a reference within a stated tolerance; (d) implement and statistically certify a sampler; (e) prove an MLE with its conditions. Practices p01, p02, p06, p07, and p13 rehearse exactly these, and p21 asks you to choose diagonal versus full covariance under constraints.

**Checkpoint 8A.** A student's batched log-density returns shape `(n,n)`. Which line most likely went wrong?

**Checkpoint 8B.** Which two practices in this unit test a log-density and a sampler under tolerances?

**Collected answers.** The quadratic form was computed as `diff @ solved` (an `(n,d) @ (d,n)` matrix product) instead of the row-wise sum `(diff * solved.T).sum(dim=1)`. p06 (log-density with `slogdet`/`solve`) and p07 (Cholesky sampling with fixed-seed statistical probes).

## 9. Forward-only deeper topic: Gaussian conditioning and mixtures

Forward-only deeper topic.
Two Gaussian facts sit just past this session.
First, conditioning one block of a Gaussian vector on another block gives another Gaussian whose mean is a linear function of the observed block; this is the engine behind Gaussian processes and Kalman filters.
Second, a **Gaussian mixture** (several Gaussians weighted by probabilities) can model clustered data that a single ellipse cannot; the unit's seeded `mixture2d` and `mixture8d` datasets are mixtures, and the VAE in Session 5 learns a continuous-latent alternative to explicit mixture components.
In `B2-023-generative-models-diffusion`, a diffusion model repeatedly applies the Section 5 rule: each forward noising step is an affine map of a Gaussian plus fresh Gaussian noise, so every intermediate distribution stays tractable.
None of these are prerequisites, permitted shortcuts, or assessed APIs in B2-022.

This final section is forward-only and not assessed.

**Checkpoint 9A.** Is Gaussian conditioning required by any B2-022 practice?

**Checkpoint 9B.** Which Section 5 rule makes each diffusion noising step tractable?

**Collected answers.** No; it is forward-only and not assessed. The affine rule: $\operatorname{Cov}(Az+b)=A\operatorname{Cov}(z)A^\top$, together with the stated fact that affine maps (and sums with independent Gaussian noise) keep vectors Gaussian.